## Tutorial 1 - Preparing a protein with Martini 3

There are a few different methods you can use to set up Martini membrane protein simulations, such as the [Martini Maker](https://www.charmm-gui.org/?doc=input/martini.bilayer) on Charmm-GUI. We will use a more flexible set up below, so you can explore all options and could potentially use the sets to automate future system assemblies if you want to do this in a high-throughput way. 

First step will be to convert our atomistic representation of a protein into a coarsse grained (CG) Martini verison, where atoms are replaced by the beads represetning that residue. To get an atomistic representation, you can get a prediction from something such as AlphaFold or use an experimentally resolved structure. Please bear in mind that the CG representation _will only be as good as your atomistic input_ so it is well worth taking time to make sure this is as accurate as possible. Possible considerations for your atomistic structure are:

<details>
<summary>    
Are there any missing loops/residues?
</summary>

It is always worth fixing missing loops if possible, and this can be done in a numebr of ways. [SwissModel](https://swissmodel.expasy.org/) is a great online sever to help with this. [Modeller](https://salilab.org/modeller/) is a locally installable tool which is great, especially for flexible loops. The predicted AlphaFold structure might also be of use to complete certain structures.

</details>

<details>
<summary>    
Protonation state of residues
</summary>

This can be an imporant consideration, and there are online tools to help with assignments of protonation states such as the [H++ server](http://newbiophysics.cs.vt.edu/H++/). The [propka tool](https://propka.readthedocs.io/en/latest/) can run on the command line for static proteins as well.

</details>

<details>
<summary>    
Are there any partner proteins?
</summary>

This very much depends on the questions you are asking, but well worth considering. If there are no experimental structures, structure prediction tools (such as AlphaFold) can be used to generate starting structures.  

</details> 

We also need to orient our protein with respect to the membrane. There are a few different tools you can use to do this, but we are going to use the [PPM webserver](https://opm.phar.umich.edu/ppm_server3_cgopm) to get our orientation. You should be able to find our protein structure that has been fixed (see above) in `tutorial_1` that you can use.  

The protein we are going to simulate today is VDAC1, in this case from a mouse (original PDB file can be found [here](https://www.rcsb.org/structure/3EMN)). On the server, most membrane proteins that are found in the PDB have already been orientated, with the example of [our protein of interest today](https://opm.phar.umich.edu/proteins/836).   

But, in the interest of show what you can do, we can also use the server to set it up. VDAC1 is a voltage dependent anion channel found in the outer membrane of mitochondria (it's a pretty cool protein if you are interested). The N-terminus of the protein in facing downwards (if we think in the z-direction) into the mitochondrial intermembrane space. We can use these bits of knowledge to set up the PPM server.  

Feel free to try this yourself, but the server might get busy. The output is found at `tutorial_1/VDAC1_ppm.pdb`   

![PPM3.0 server options](images/PPM_inputs.png)

In [2]:
!pwd

/storage/lfsmgr_grp/nttpkm/CCPBioSim_training/tutorial_1


Now we have an oriented protein! We can check this with VMD 

Do this **out of the notebook** in your own terminal:

```$vmd VDAC1_ppm.pdb```

While the dummy beads (DUM) are great for showing where the membrane should be, we need to remove it before our next steps. We can do that using a simple command:

In [1]:
!grep -v DU VDAC1_ppm.pdb > VDAC1_clean.pdb

We are now ready to convert our protein to a Martini 3 representation. Let's have a look at the possible options when using [martinize](https://elifesciences.org/reviewed-preprints/90627), which is the tool we are going to use to convert between AT and CG resolutions: 

In [5]:
!martinize2 -h

usage: martinize2 [-h] [-V] [-f INPATH] [-x OUTPATH] [-o TOP_PATH] [-sep]
                  [-merge MERGE_CHAINS] [-name MOLNAME]
                  [-resid RESID_HANDLING]
                  [-ignore IGNORE_RES [IGNORE_RES ...]] [-ignh]
                  [-model MODELIDX] [-bonds-from {name,distance,none,both}]
                  [-bonds-fudge BONDS_FUDGE] [-ff TO_FF] [-from FROM_FF]
                  [-ff-dir EXTRA_FF_DIR] [-map-dir EXTRA_MAP_DIR] [-list-ff]
                  [-list-blocks] [-p {none,all,backbone}] [-pf POSRES_FC]
                  [-dssp [DSSP] | -ss SEQUENCE | -collagen] [-ed] [-elastic]
                  [-ef RB_FORCE_CONSTANT] [-el RB_LOWER_BOUND]
                  [-eu RB_UPPER_BOUND] [-ermd RES_MIN_DIST]
                  [-ea RB_DECAY_FACTOR] [-ep RB_DECAY_POWER]
                  [-em RB_MINIMUM_FORCE] [-eb RB_SELECTION] [-eunit RB_UNIT]
                  [-go [GO]] [-go-eps GO_EPS] [-go-low GO_LOW] [-go-up GO_UP]
                  [-go-res-dist GO_RES_DIST] [-g

There are many options for the input here, but we will focus on the ones used in this tutorial:

- `-f` which specifies the input file for your protein. Need to make sure you have removed anything that isn't protein
- `-x` the output coordinate file for the martini coordinate file
- `-o` the topology file that will be created, this points towards any .itp files that are created
- `-name` this sets the name of the protein, and is not strictly needed. This becomes useful when simulating multi-protein systems
- `-dssp` the flag used to determine structural features, such as alpha helcies/beta sheets etc, so the correct parameters can be used. An executable pointing towards a way to run dssp can be specified after this flag, otherwise mdtraj will be used
- `-elastic` this flag will ensure elastic bonds are written. Without this (or the use of GoMartini) there will be no secondary structure retention and the protein will unfold without the use of other restraints
- `-ef` the force constant used for the elastic network (i.e. how strong the elastic bands are) in kJ mol<sup>-1</sup> nm<sup>-2</sup>
- `-el` the lower boundary for elastic networks (in nm)
- `-eu` the upper boundary for elastic networks (in nm)


The default values are normally a good place to start for the flags that specify a value. See if you can put together a martinize2 command to convert our protein from AT to CG resolution:

In [ ]:
!martinize2 -f VDAC1_clean.pdb ....

<details>
<summary>    
<i>Really</i> stuck? Click on this to reveal a command we can use
</summary>

`!martinize2 -f VDAC1_clean.pdb -x VDAC1_cg.pdb -o topol.top -name VDAC1 -dssp -elastic -ef 700 -el 0 -eu 0.8`  

If there's anything you don't understand please ask!

</details>

Other commands that might be useful with martinize with your own systems:

- `-merge` if you have multiple protein chains (in a complex for example) this treats them as one for coarse-graining, and will make elastic bonds between the different chains to retain tertary structure
- `-id-regions` which applies the new [Martini3-IDP forcefield](https://www.nature.com/articles/s41467-025-58199-2) to the specified regions so they have behaviour that resembles intriniscally disordered (ID) proteins. This ensures there are not elastic bonds in this region
- `-go` (and related flags) will construct a [GoMartini](https://www.nature.com/articles/s41467-025-58719-0) network, which would be used _instead_ of an elastic network to retain secondary structure. This method is slightly more computationally expensive, but can capture conformational rearrangements
- `-modify` this lets us make changes to the protein, such as mutations, but also alter the protonation state of any important residues.


Now let's visulize our protein!

Do this **out of the notebook** in your own terminal:

```$vmd your_CG_structure.pdb```

### Add an image here for visulising the protein, maybe with elastic networks

<details>
<summary>    
<b>Question: how do you think you could evaluate if you are using a good force constant value?</b>
</summary>

One method is to compare the root mean squared fluctuation (RMSF) of an atomistic simulation of your protein with a CG simulation. If regions are too flexible/rigid you can then change the force constant and the cutoffs used! This depends on how important capturing the flexibility of the protein is to the question you are asking, the default values <i>do a pretty good job</i> for many systems.

</details>